# Big Data Platforms — Lecture 5 Notes

Course: DATA140031 (MOOC, 3 ECTS) + DATA140032 (MOOC Exam, 2 ECTS)
Lecturer: Keijo Heljanko, Department of Computer Science, University of Helsinki
Date: 15.9.2026

Lecture 5 opens with two worked exercises that put real numbers behind ideas from
lecture 4 — just how fragile RAID 0 actually is at scale, and how real the URE
risk during a RAID 5 rebuild turns out to be. From there it moves up a level, away
from RAID specifically and into the broader storage hierarchy: RAM, Flash, hard
disk, and tape, what each is actually good and bad at, and how a real system (the
lecture uses Facebook's storage stack as the worked example) ends up mixing all
four rather than picking just one. The two worked exercises are reproduced as
runnable code below rather than just algebra on a slide, since re-deriving the
numbers is a good way to actually internalise them.


## 1. Worked exercise — fault tolerance of RAID 0

**Question:** assume a mean time to failure (MTTF) of a single hard disk of
300,000 hours. Using 4 TB disks in a RAID 0 array (striped, no redundancy)
totalling 64 TB of storage, what's the MTTF of the *array* in years?

**Hint given in the lecture:** the array fails the moment *any one* disk in it
fails.

### Setting up the model

Each disk's failures are modelled as a **Poisson process** — failures happen at
random, at some constant average rate λ, independently of how long the disk has
already survived. (This is a simplification — real disks don't fail at a
perfectly constant rate over their whole life — but it's the standard first-order
model, and it's exactly what "MTTF" as a single number implicitly assumes.)

A convenient property of Poisson processes: if you have *K* independent
processes each firing at rate λ, the *combined* process — "does any one of them
fire" — is itself Poisson, with rate *K·λ*. That's precisely the RAID 0 failure
model: the array's failure rate is just the sum of its disks' individual failure
rates, because losing *any single* disk kills the whole array.

### Working through the numbers


In [1]:
hours_per_year = 300_000 / 300_000  # placeholder just to anchor units below

# Given
disk_mttf_hours = 300_000          # MTTF of a single hard disk, in hours
disk_capacity_tb = 4                # capacity of a single disk, in TB
array_capacity_tb = 64              # total desired RAID 0 array capacity, in TB
hours_per_year = 365.244 * 24       # matches the lecture's figure (accounts for leap years)

# Step 1: how many disks does the array need?
K = array_capacity_tb / disk_capacity_tb
print(f"K = {K:.0f} disks")

# Step 2: single-disk failure rate lambda, in failures/year
disk_mttf_years = disk_mttf_hours / hours_per_year
lam = 1 / disk_mttf_years
print(f"Single-disk MTTF  = {disk_mttf_years:.3f} years")
print(f"Single-disk lambda = {lam:.5f} failures/year")

# Step 3: array failure rate F = K * lambda (RAID 0: any one disk failing kills the array)
F = K * lam
print(f"\nArray failure rate F = K * lambda = {F:.5f} failures/year")

# Step 4: array MTTF is just 1/F
array_mttf_years = 1 / F
print(f"Array MTTF = 1 / F = {array_mttf_years:.2f} years")


K = 16 disks
Single-disk MTTF  = 34.224 years
Single-disk lambda = 0.02922 failures/year

Array failure rate F = K * lambda = 0.46751 failures/year
Array MTTF = 1 / F = 2.14 years


### Reading the result

A 64 TB RAID 0 array built from 4 TB disks (so 16 disks total) has an MTTF of
roughly **2.14 years** — a full order of magnitude worse than any individual
disk's 34-year MTTF. That's the whole point of the exercise: striping with zero
redundancy doesn't just fail to *improve* reliability, it actively multiplies the
failure rate by the number of disks in the array. Every extra disk you add for
capacity is one more independent way for the whole array to die. This is exactly
why lecture 4 flagged RAID 0 as suitable only for scratch data you're fine losing
— the math here shows precisely how fast that risk compounds as an array grows.


## 2. Worked exercise — URE risk during a RAID 5 rebuild

**Question:** using consumer-grade hard disks with a typical unrecoverable read
error (URE) rate of 1 error per 10¹⁵ bits read, what's the *expected number* of
URE errors during the rebuild of a full 16 TB RAID 5 array?

**Hint given in the lecture:** rebuilding reads back as much data as the array
actually holds — in the worst case, that's the array's entire storage capacity.

Recall from lecture 4 exactly why this matters: if a URE happens on *any* other
disk while a RAID 5 array is already rebuilding from one failed disk, that's a
second failure the array can't tolerate, and the rebuild fails outright.


In [2]:
bits_per_byte = 8

# Given
array_capacity_tb = 16
ure_rate = 1 / 10**15   # 1 unrecoverable read error expected per this many bits read

# Step 1: total bits that must be read during the rebuild
array_capacity_bytes = array_capacity_tb * 10**12   # decimal TB, matching drive-vendor convention
total_bits_read = array_capacity_bytes * bits_per_byte
print(f"Total bits read during rebuild: {total_bits_read:.3e} bits")

# Step 2: expected number of UREs = bits read * URE rate
expected_ures = total_bits_read * ure_rate
print(f"Expected number of UREs during rebuild: {expected_ures:.3f}")


Total bits read during rebuild: 1.280e+14 bits
Expected number of UREs during rebuild: 0.128


### Reading the result

The expected number of UREs during this particular rebuild comes out to
**0.128** — comfortably under 1, which sounds reassuring on its own. But this is
an *expected value*, not a guarantee, and it scales directly with array size: a
32 TB array built the same way would already sit around 0.26 expected UREs, and
consumer drives commonly ship in 16+ TB capacities today, with RAID 5 arrays
routinely spanning many times that. The trend across the whole industry has been
disk *capacity* growing much faster than disk *URE rate* improving — which is
exactly the argument lecture 4 already made for preferring RAID 6 (survives two
failures) or erasure coding over RAID 5 at any serious scale: the "expected
number of failures during rebuild" keeps creeping toward 1 as arrays get bigger,
even while each individual disk's own reliability spec stays roughly the same.


In [3]:
# A quick sensitivity check: how does the expected URE count during rebuild
# scale as the array gets bigger, holding the disk's URE rate fixed?

for capacity_tb in [8, 16, 32, 64, 128, 256]:
    total_bits = capacity_tb * 10**12 * bits_per_byte
    expected = total_bits * ure_rate
    print(f"{capacity_tb:>4} TB array -> expected UREs during full rebuild: {expected:.3f}")


   8 TB array -> expected UREs during full rebuild: 0.064
  16 TB array -> expected UREs during full rebuild: 0.128
  32 TB array -> expected UREs during full rebuild: 0.256
  64 TB array -> expected UREs during full rebuild: 0.512
 128 TB array -> expected UREs during full rebuild: 1.024
 256 TB array -> expected UREs during full rebuild: 2.048


## 3. Storage technologies for the cloud

Zooming out from RAID specifically, the lecture lists four storage technologies
that actually show up in cloud infrastructure: **RAM**, **Flash (SSD)**,
**hard disk**, and **tape**. The rest of the lecture is mostly about
understanding the trade-offs between these four and picking the right one — or
more realistically, the right *mix* — for a given workload.


## 4. Flash storage

Flash (SSD) has been steadily displacing hard disks across many applications,
and the trend lines explain why: Flash capacity per euro has been improving
*faster* than hard disk capacity per euro, and Flash already accounts for
roughly **70% of all enterprise storage capacity**, still growing. Random-access
IOPS is where the gap is most dramatic — SSDs can hit **1,000–10,000×** the
random read/write IOPS of even high-end hard disks, and sequential throughput
also comfortably beats the best hard disks. With no moving parts, SSDs also fail
less often mechanically than spinning disks, which combined with strong random
I/O performance makes them an excellent match for typical laptop and desktop
usage patterns.

**But Flash is not a drop-in hard disk replacement for every server workload.**
The catch is **write endurance** — the total amount of data an SSD is specified
to reliably write over its lifetime — which is often *much* smaller than a hard
disk's effective lifetime write budget. For genuinely write-intensive server
workloads, hard disks can still be the more economically sensible choice,
precisely because an SSD under that load may need replacing once its write
endurance is exhausted, well before a comparable hard disk would need replacing.
Put the two properties together: Flash's *failures* are rare, but its *end of
life via wear* is a real, predictable, and separate concern from ordinary
hardware failure.


## 5. Flash organization — why writes are the awkward part

The read/write asymmetry above isn't incidental — it comes directly from how
flash chips are physically organized, and it's worth understanding the mechanism
to see why "write endurance" is a real physical limit and not just a marketing
number.

- A flash chip is organized into **pages**, typically a few KB each (e.g. 2KB).
- A **page read** loads that page's data into a buffer that can then be accessed
  quickly and randomly — this is the fast, cheap operation.
- A **page write** can only flip bits in one direction: from **0 to 1**. It
  cannot flip a bit back from 1 to 0.
- To flip bits from 1 back to 0, an entirely separate **erase** operation is
  required — and erase doesn't operate at the page level, it operates on a much
  larger **block** (commonly around 128KB, i.e. many pages at once).
- If that block being erased still contains *any* live data, that data has to be
  read out and rewritten somewhere else *first*, before the erase can happen —
  otherwise it's simply gone.

So a small, in-place-looking modification to a flash drive can actually trigger
reading and rewriting an entire block's worth of unrelated data elsewhere, just
to make room for the erase. This is exactly the physical reason writes are
slower and more "expensive" (in wear terms) than reads on flash, in a way that
has no real analogue on a spinning disk.


## 6. Flash types and endurance

NAND flash — the dominant variety — comes in several cell types, trading cost
against write endurance:

| Type | Relative cost | Typical erase cycles per block |
|---|---|---|
| QLC (quad-level cell) | Cheapest | ~1,000 |
| TLC (triple-level cell) | Second cheapest | ~1,000 |
| MLC (multi-level cell) | Cheap | 1,000–10,000 |
| SLC (single-level cell) | Expensive | 100,000–1,000,000 |

The pattern is a clean trade-off: packing more bits into each cell (QLC's
"quad-level" stores 4 bits per cell, versus SLC's single bit) drives the cost per
gigabyte down, but each additional bit-per-cell level makes the cell's voltage
states harder to distinguish reliably after repeated erase cycles, which is what
drags the endurance rating down as you move from SLC toward QLC.


In [4]:
# A rough illustration of how these endurance figures translate into
# real-world write budgets over a drive's life -- not a precise model of any
# particular controller, just enough to make "write endurance" concrete.

drive_capacity_tb = 1
bytes_per_tb = 10**12

flash_types = {
    "QLC": 1_000,
    "TLC": 1_000,
    "MLC": 5_000,      # midpoint of the 1,000-10,000 range given in the lecture
    "SLC": 500_000,    # midpoint of the 100,000-1,000,000 range given in the lecture
}

for name, erase_cycles in flash_types.items():
    # Total lifetime bytes the whole drive could absorb if writes were spread
    # perfectly evenly across every block (ideal wear levelling).
    lifetime_writes_tb = drive_capacity_tb * erase_cycles
    print(f"{name}: ~{erase_cycles:>7,} erase cycles -> "
          f"~{lifetime_writes_tb:,} TB total writes over the drive's life "
          f"(for a {drive_capacity_tb} TB drive, ideal wear levelling)")


QLC: ~  1,000 erase cycles -> ~1,000 TB total writes over the drive's life (for a 1 TB drive, ideal wear levelling)
TLC: ~  1,000 erase cycles -> ~1,000 TB total writes over the drive's life (for a 1 TB drive, ideal wear levelling)
MLC: ~  5,000 erase cycles -> ~5,000 TB total writes over the drive's life (for a 1 TB drive, ideal wear levelling)
SLC: ~500,000 erase cycles -> ~500,000 TB total writes over the drive's life (for a 1 TB drive, ideal wear levelling)


## 7. Wear levelling

Given that endurance is finite and per-block, SSD controllers run sophisticated
**wear levelling** algorithms whose entire job is to spread writes as evenly as
possible across every physical block on the drive — so that no single "hot"
block wears out from erase cycles while the rest of the drive sits nearly
untouched.

One practical side effect: as a flash drive fills up, its remaining free space
can become fragmented across many partially-used blocks. Since erase always
operates on whole blocks, the controller sometimes has to shuffle live data
around just to free up a contiguous block to write into — this is *extra*
write traffic the drive generates on its own, on top of whatever the host
actually asked it to write. To soften this, SSDs commonly reserve **spare
capacity** — extra physical space beyond what's advertised as usable — purely to
give the wear-levelling and garbage-collection logic room to work without
constantly running into fragmentation.


## 8. Sustained write performance can drop under load

A flash drive's headline write-speed number is typically measured on a fresh,
mostly-empty drive with plenty of spare capacity and an empty write cache to
absorb bursts. Under a genuinely *sustained* heavy write workload, that cache
fills up and the wear-levelling/garbage-collection machinery has to keep pace in
real time — and write throughput can drop substantially as a result, sometimes
to a fraction of the initial burst rate.

Reference cited in the lecture (an independent SSD benchmark showing exactly
this precondition curve across several drives):
<http://www.storagereview.com/samsung_ssd_840_pro_review>


### Sustained flash write throughput vs. time
![Sustained flash write throughput vs. time ](images/flashtime.png)


The practical takeaway: benchmark numbers taken over a short burst can be
badly misleading for a workload that writes continuously for hours. If a
workload is genuinely write-heavy and sustained, it's worth specifically
checking (or benchmarking) *sustained* write performance, not just the headline
peak figure.


## 9. Optimizing for flash in practice

Given everything above, a few practical guidelines fall out naturally:

- **Minimize the total number of bytes written to flash.** This directly extends
  the drive's usable lifetime (less wear), and it also *improves* ongoing
  performance, since fewer writes mean less frequent — and slow — block erases
  competing with the workload's own I/O.
- **Heavy sequential write workloads are a specific danger zone** — database
  write-ahead logs are the lecture's example. These generate a continuous stream
  of writes that can burn through write endurance surprisingly fast, precisely
  because sequential logging is, by design, meant to write constantly.
- **The TRIM command matters.** TRIM lets the operating system tell the SSD
  which previously-written blocks are now free (because the file that used them
  was deleted), so the drive's own garbage collection can reclaim and erase
  those blocks proactively, ahead of actually needing the space. Without TRIM,
  the drive has no way to know a block is free until something tries to write
  over it, which pushes wear-levelling and erase work into the critical path of
  that later write instead of happening calmly in the background.


## 10. Rule-of-thumb latency numbers

These are the numbers that explain *why* the storage hierarchy is shaped the way
it is — everything downstream in this lecture (caching strategy, RAMCloud,
Facebook's hot/warm/cold tiers) ultimately traces back to these three figures.
For a single random-access read:

| Medium | Latency | Relative to RAM |
|---|---|---|
| RAM memory reference | 100 ns | 1× (baseline) |
| Flash drive access | 100,000 ns | ~1,000× slower |
| Disk seek | 10,000,000 ns | ~100,000× slower |


In [5]:
ram_ns = 100
flash_ns = 100_000
disk_ns = 10_000_000

print(f"Flash is {flash_ns / ram_ns:,.0f}x slower than RAM")
print(f"Disk seek is {disk_ns / ram_ns:,.0f}x slower than RAM")
print(f"Disk seek is {disk_ns / flash_ns:,.0f}x slower than a flash access")


Flash is 1,000x slower than RAM
Disk seek is 100,000x slower than RAM
Disk seek is 100x slower than a flash access


Each step down the hierarchy is roughly **two to three orders of
magnitude** slower than the one above it — this isn't a gentle gradient, it's a
series of cliffs. That's the entire justification for caching: any technique
that turns a disk seek into a RAM hit is worth a great deal, because the two are
separated by five orders of magnitude, not a small constant factor.


## 11. Storage price trends over time

Prices for RAM, Flash, and hard disks have all fallen dramatically over the
decades — but not at the same rate, and disk has consistently stayed the
cheapest per-terabyte option, with Flash closing the gap faster than RAM has.


### Historical price of computer memory and storage (log scale, $/TB)

![Historical price of computer memory and storage (log scale, $/TB) ](images/storage_price.png)


One framing worth internalising from the lecture: **RAM today is roughly
the same price hard disks were about a decade ago.** That single observation
does a lot of work — it means workloads that only became economically viable on
hard disks a decade back are now perfectly viable to run entirely in RAM today.
Turned around, that gives a genuinely useful design heuristic: **hard-disk-based
storage should be reserved specifically for datasets that aren't yet
economically viable to keep in RAM (or Flash)** — which is really a definition of
what "Big Data" (in the HDD-scale sense) actually *is*: workloads that weren't
economically feasible on hard disks a decade ago either, and have only recently
become affordable to store and process at all, on any medium.


## 12. Storage usage scenarios: where each technology still fits

- **Tape** is still very much alive for **backup** purposes specifically — its
  cost per terabyte remains hard to beat for data that's written once and then
  mostly just needs to sit there, rarely touched, for a long time.
- **RAM and Flash should be used wherever the economics allow**, given how much
  faster both are than hard disk for random access, per the latency table above.
- **Hard disk remains the right tool specifically for datasets too large to be
  economical in RAM or Flash yet** — which, again, shifts every year as RAM and
  Flash prices keep falling.


## 13. Total cost of ownership: RAM vs. Flash vs. Disk

Ousterhout et al.'s *The case for RAMCloud* (Communications of the ACM 54(7):
121–130, 2011) works through exactly this trade-off as a function of two
variables: how much data you have, and how many queries per second you need to
serve against it.


### Total cost of ownership by dataset size and query rate — RAM vs Flash vs Disk
![Total cost of ownership by dataset size and query rate — RAM vs Flash vs Disk](images/historical_price.png)


The pattern the chart makes visible: **large dataset + low query rate**
favours disk (you're paying mostly for raw capacity, and don't need much random
I/O), while **small dataset + high query rate** favours RAM (capacity cost is
trivial, but you need every IOPS you can get), with Flash occupying the middle
ground between the two. This maps directly onto the "temperature" framing the
lecture introduces next.

### Hot, warm, and cold storage

- **Hot storage** — high IOPS-per-TB requirement. RAM is the cheapest way to buy
  that much random-access throughput, even though it's the most expensive medium
  per raw terabyte.
- **Cold storage** — high TB-per-IOPS requirement, i.e. lots of capacity needed,
  but relatively little random access. Hard disks are the cheapest way to buy
  raw capacity at that ratio.
- **Warm storage** — Flash sits as the compromise point between the two,
  offering more IOPS per TB than disk and more TB per euro than RAM.

The genuinely important caveat: **these boundaries move whenever relative
pricing moves.** There's no fixed rule saying "always use Flash for X" — the
optimal choice for a given dataset size and query rate is a moving target that
tracks the current relative prices of RAM, Flash, and disk, which is exactly why
this is framed as a total-cost-of-ownership calculation rather than a fixed
lookup table.


## 14. Worked example: Facebook's storage tiers

Facebook's own research into this — Muralidhar et al., *f4: Facebook's Warm BLOB
Storage System* — measured exactly how the "temperature" (access frequency) of a
piece of stored data evolves as it ages.


###Facebook BLOB request rate and IOPS/TB by age
![Facebook BLOB request rate and IOPS/TB by age](Images/facebook_blob.png)


**The finding, in one sentence: data at Facebook cools as it ages.** A
photo or video that's brand new gets accessed constantly — friends and family
are actively looking at it — and that request rate falls off sharply, often by
multiple orders of magnitude, within days to weeks. This isn't a minor effect;
the request-rate charts show drops of a full order of magnitude at several
distinct points as content ages from a day old to months old to years old.

Because the *optimal* storage medium for a given file depends on how often it's
actually being accessed (per the hot/warm/cold framing above), and that access
rate changes predictably and dramatically over a file's life, Facebook runs
**separate storage systems tuned for hot, warm, and cold data respectively**, and
actively **migrates files from one tier to another as they age and cool** — new
uploads land in the hot tier, and get moved down to warm and eventually cold
storage as their measured request rate drops, purely to keep costs down while
keeping frequently-accessed content fast.


## 15. Jim Gray's storage mantra (2006)

The lecture closes out the storage-hierarchy discussion by quoting a 2006
presentation from Turing Award winner **Jim Gray** of Microsoft
(<http://research.microsoft.com/en-us/um/people/gray/talks/Flash_is_Good.ppt>).
What's striking is how well this held up nearly two decades later:

> Tape is Dead. Disk is Tape. Flash is Disk. RAM Locality is King.

The idea behind each step: whatever role the *previous* medium used to play in
the hierarchy, the *next* medium down effectively inherits it, as prices and
capabilities shift the whole stack down one rung. Tape's old niche (bulk,
rarely-accessed cold storage) is increasingly what disk is used for. Disk's old
niche (general-purpose active storage) is increasingly what Flash handles. And
what actually determines performance more than anything else is whether your
*working set* has locality — whether the data you need right now tends to
already be sitting in RAM.

### Gray's 2006 view of disk
- Cheap per unit capacity.
- A full **sequential** read of a disk takes hours.
- A full **random-access** read of a disk takes *weeks* — an enormous gap
  between the two access patterns on the same physical medium.
- Consequence: disk should mostly be treated as a **cold-storage archive**, not
  as a medium you expect to hit randomly at any real rate.

### Gray's 2006 view of flash
- Lots of IOPS.
- Expensive compared to disk (though improving — and this has borne out
  dramatically since 2006).
- Limited write endurance.
- Slower to write than to read.

### Gray's 2006 view of RAM
- Flash/disk sits **100,000–1,000,000 CPU cycles** away, in terms of latency.
- RAM sits only **~100 CPU cycles** away.
- Gray's conclusion from that gap alone: **main-memory databases were going to
  become common** — and they have, DRAM-first systems are now an entirely
  ordinary category of database rather than a research curiosity.


In [6]:
cpu_cycles_ram = 100
cpu_cycles_flash_disk_low = 100_000
cpu_cycles_flash_disk_high = 1_000_000

print(f"Flash/disk is between {cpu_cycles_flash_disk_low / cpu_cycles_ram:,.0f}x "
      f"and {cpu_cycles_flash_disk_high / cpu_cycles_ram:,.0f}x further from the "
      f"CPU (in cycles) than RAM is.")


Flash/disk is between 1,000x and 10,000x further from the CPU (in cycles) than RAM is.


## 16. Caching to improve hard-disk performance

The obvious response to disk's latency problem is to cache the hot portion of a
dataset in RAM. It's a well-known and effective technique — but the sheer size
of the RAM/disk latency gap means a cache has to be *extremely* effective (very
high hit rate) before it actually helps much overall, because every single miss
still costs the full disk-seek penalty.

### RAMCloud's worked numbers

Ousterhout et al.'s RAMCloud paper puts real figures behind exactly this point,
using Facebook's own 2009 storage profile as the example. Facebook at the time
ran with enough RAM to fit roughly **75%** of their dataset (excluding images and
video) entirely in memory. The paper then asks: what if they pushed the RAM
cache hit rate up further, to 99%?


In [7]:
ram_ns = 100
disk_seek_ns = 10_000_000
flash_ns = 100_000

def average_latency_ns(hit_rate, miss_penalty_ns, hit_ns=ram_ns):
    return hit_rate * hit_ns + (1 - hit_rate) * miss_penalty_ns

# Scenario from the slides: 99% RAM cache hit rate, misses go to a hard disk seek
avg_disk_backed = average_latency_ns(0.99, disk_seek_ns)
print(f"99% RAM hit rate, disk-backed misses:  average latency = {avg_disk_backed:,.0f} ns")
print(f"  -> that's {avg_disk_backed / ram_ns:,.1f}x worse than a pure RAM hit\n")

# Scenario from the slides: same 99% hit rate, but misses go to Flash instead
avg_flash_backed = average_latency_ns(0.99, flash_ns)
print(f"99% RAM hit rate, flash-backed misses: average latency = {avg_flash_backed:,.0f} ns")
print(f"  -> that's {avg_flash_backed / ram_ns:,.1f}x worse than a pure RAM hit")


99% RAM hit rate, disk-backed misses:  average latency = 100,099 ns
  -> that's 1,001.0x worse than a pure RAM hit

99% RAM hit rate, flash-backed misses: average latency = 1,099 ns
  -> that's 11.0x worse than a pure RAM hit


### Reading the result

Even at a 99% RAM cache hit rate — which sounds like an excellent cache by any
normal standard — backing the remaining 1% of misses with a hard-disk seek
still produces an *average* latency around **100,099 ns**, more than a
thousand times worse than pure RAM. Swapping the backing store for Flash instead
of disk brings that down to roughly **1,099 ns** — better, but still about
**10× worse than 100% RAM**.

The core insight: this isn't primarily a throughput argument, it's a **latency**
argument, though the paper notes the same reasoning carries over to aggregate
IOPS numbers too. **If a system can afford enough memory to hold 100% of its
working set in RAM** — rather than relying on a cache with inevitable, rare-but-
catastrophically-expensive misses down to Flash or disk — it gets dramatically
better latency *and* can serve more IOPS from the same number of servers. That's
the entire pitch behind RAMCloud-style architectures: don't cache the hot data in
RAM, just *keep the whole working set there* and treat anything else as a
different problem entirely.

### What actually limits RAMCloud-style designs

Two genuinely hard problems stand between "keep everything in RAM" and a
production system:

1. **Durability and fast recovery.** RAM is volatile — a power failure or server
   crash loses everything held in it, so a RAMCloud-style system needs some
   mechanism to guarantee data survives a crash and can be recovered quickly
   afterward, without falling back to disk-speed recovery in the process.
2. **Network hardware and OS overhead low enough to actually keep RAM busy.**
   Once local memory access drops to ~100ns, the network path to a *remote*
   server's RAM becomes the new bottleneck unless the networking stack itself is
   built for comparably low latency.

Further reading on that second point specifically: Luiz Barroso et al.,
*Attack of the killer microseconds*, Communications of the ACM, Vol. 60, Issue 4
(April 2017), pp. 48–54. <https://doi.org/10.1145/3015146>


## 17. Improving disk-based database performance when RAM can't hold it all

Sometimes a database genuinely needs to be far larger than any affordable amount
of RAM — RAMCloud's "put everything in memory" approach simply isn't an option
at that scale. RAM being an expensive resource per byte means it has to be spent
carefully rather than just thrown at the whole dataset. A few complementary
techniques:

- **Cache the hot subset of data in RAM.** The straightforward approach — this
  directly improves latency for whichever items happen to already be sitting in
  RAM when they're requested.
- **Use RAM to speed up queries for items that *aren't* in the database at
  all**, not just ones that are. If a lookup is destined to come back empty, it's
  far better to know that without ever touching disk.
  - One option: hold a full **index** of everything on disk, entirely in RAM.
    The problem is that the index itself can grow large enough that it no
    longer fits in RAM either, especially as the underlying dataset scales up.
  - A better option at scale: **probabilistic data structures**, which trade a
    small, tunable false-positive rate for a dramatic reduction in memory
    footprint compared to an exact index. The canonical example is the
    **Bloom filter** — it can answer "is this item *definitely not* in the
    dataset?" using a small fraction of the memory a full index would need,
    letting a system skip an expensive disk lookup entirely for the (common)
    case of a query that's going to come back empty anyway.


In [8]:
# A minimal illustration of a Bloom filter, purely to make the "small memory
# footprint, no false negatives, some false positives" trade-off concrete.
# Not production code -- a real implementation would use good hash functions
# and size the bit array based on expected item count and target false-
# positive rate.

import hashlib

class TinyBloomFilter:
    def __init__(self, size_bits=64, num_hashes=3):
        self.size_bits = size_bits
        self.num_hashes = num_hashes
        self.bits = [False] * size_bits

    def _positions(self, item):
        for i in range(self.num_hashes):
            digest = hashlib.sha256(f"{i}:{item}".encode()).hexdigest()
            yield int(digest, 16) % self.size_bits

    def add(self, item):
        for pos in self._positions(item):
            self.bits[pos] = True

    def might_contain(self, item):
        # If ANY of this item's bits are unset, it is DEFINITELY not present.
        # If ALL of its bits are set, it PROBABLY is present (or a false positive).
        return all(self.bits[pos] for pos in self._positions(item))


bf = TinyBloomFilter(size_bits=64, num_hashes=3)
on_disk_items = ["order_1001", "order_1002", "order_1003"]
for item in on_disk_items:
    bf.add(item)

test_items = ["order_1001", "order_9999", "order_4242"]
for item in test_items:
    on_disk = item in on_disk_items
    filter_says = bf.might_contain(item)
    verdict = "definitely absent, skip disk lookup" if not filter_says else \
              "might be present, worth checking disk"
    print(f"{item:>12}: actually on disk = {on_disk!s:<5}  "
          f"filter verdict = {filter_says!s:<5}  ({verdict})")


  order_1001: actually on disk = True   filter verdict = True   (might be present, worth checking disk)
  order_9999: actually on disk = False  filter verdict = False  (definitely absent, skip disk lookup)
  order_4242: actually on disk = False  filter verdict = False  (definitely absent, skip disk lookup)


The filter never produces a false *negative* — if it says "definitely
not present," that's a guarantee, and the disk lookup can be safely skipped
entirely. It can occasionally produce a false *positive* — saying "might be
present" for something that actually isn't — which just costs one wasted disk
lookup, not a wrong answer. That asymmetry is exactly what makes Bloom filters
useful here: skip the disk when the filter is confident, and fall back to the
(comparatively expensive) real lookup only when it isn't.


## Summary

This lecture connects two levels of the storage story that earlier lectures kept
mostly separate. First, the worked RAID exercises put hard numbers behind
lecture 4's warnings: striping without redundancy doesn't just fail to help
reliability, it multiplies the failure rate by the disk count, and the
probability of hitting a URE during a rebuild — while individually small — scales
directly with array size in a way that keeps eroding RAID 5's safety margin as
drives get bigger. Second, zooming out to RAM/Flash/disk/tape as a whole, the
recurring theme is that **there is no single best storage medium** — only a
medium that's currently cheapest for a given combination of capacity needed and
access rate required, and that combination is a moving target as relative prices
keep shifting (RAM today costs what disk cost a decade ago) and as any given
piece of data's own access pattern cools with age. Facebook's tiered hot/warm/cold
storage and RAMCloud's "just keep the whole working set in RAM" approach are two
different, equally rational responses to the same three numbers — 100ns, 100µs,
and 10ms — that open this lecture's discussion of the storage hierarchy.
